In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# load data
data = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# see data
print(data.head()) # shows the first 5 rows of the DataFrame to understand in a quick way how the dataset looks like

# feature engineering: # feature engineering begins here
# In regression tasks:X = features (input), y = target (output)

# Fix budget (replace 0 values)
# budget = 0 is suspicious because movies usually do not have an actual budget of 0
# so we replace 0 with the median budget value
data["budget"] = data["budget"].replace(0, data["budget"].median())
test["budget"] = test["budget"].replace(0, test["budget"].median())

# Log transform budget
# movie budgets are highly skewed because blockbuster movies can have extremely large budgets
# log transformation compresses large values and stabilizes the distribution
data["budget_log"] = np.log1p(data["budget"])
test["budget_log"] = np.log1p(test["budget"])

# Parse dates
data["release_date"] = pd.to_datetime(data["release_date"], errors="coerce", format="mixed")
test["release_date"] = pd.to_datetime(test["release_date"], errors="coerce", format="mixed")

# Extract year
# release year may capture market changes and long-term movie industry trends
data["release_year"] = data["release_date"].dt.year
test["release_year"] = test["release_date"].dt.year

# Extract release month
# some movies perform better in certain seasons like summer or christmas
data["release_month"] = data["release_date"].dt.month
test["release_month"] = test["release_date"].dt.month



# add feature cast_size

def get_cast_size(x):
    try:
        cast = ast.literal_eval(x)
        return len(cast)
    except (ValueError, SyntaxError, TypeError):
        return 0

# larger casts may indicate bigger productions and blockbuster movies
data["cast_size"] = data["cast"].apply(get_cast_size)
test["cast_size"] = test["cast"].apply(get_cast_size)


# add feature interaction
# combines budget and popularity into one feature
# expensive and popular movies may perform especially well
data["budget_popularity_ratio"] = data["budget_log"] * data["popularity"]
test["budget_popularity_ratio"] = test["budget_log"] * test["popularity"]



def get_names(x):
    try:
        items = ast.literal_eval(x)
        return [item["name"] for item in items]
    except (ValueError, SyntaxError, TypeError):
        return []

# genre count
data["genre_count"] = data["genres"].apply(
    lambda x: len(get_names(x))
)
test["genre_count"] = test["genres"].apply(
    lambda x: len(get_names(x))
)

# keyword count
data["keyword_count"] = data["Keywords"].apply(
    lambda x: len(get_names(x))
)
test["keyword_count"] = test["Keywords"].apply(
    lambda x: len(get_names(x))
)

# disney feature
data["top_disney"] = data["production_companies"].apply(
    lambda x: int("Disney" in get_names(x))
)
test["top_disney"] = test["production_companies"].apply(
    lambda x: int("Disney" in get_names(x))
)



# add director feature
def get_director(x):
    try:
        crew = ast.literal_eval(x)
        for person in crew:
            if person.get("job") == "Director":
                return person["name"]
        return "unknown"
    except (ValueError, SyntaxError, TypeError):
        return "unknown"

data["director"] = data["crew"].apply(get_director)
test["director"] = test["crew"].apply(get_director)

# director_freq counts how often each director appears in the dataset
# more frequent directors may indicate more experienced or commercially active filmmakers
# this feature captures popularity/experience without using target information (no leakage)
director_freq = data["director"].value_counts()

data["director_freq"] = data["director"].map(director_freq)
test["director_freq"] = test["director"].map(director_freq)

# unseen directors in the test set are filled with 0 frequency
test["director_freq"] = test["director_freq"].fillna(0)





# define features and target

features = [
    "budget_log",
    "popularity",
    "runtime",
    "release_year",
    "release_month",
    "genre_count",
    "cast_size",
    "budget_popularity_ratio"
]

features.append("keyword_count")

features.append("top_disney")
features.append("director_freq")


# we tell python we only care about these columns for now

X = data[features]
# Creates a table X with only the selected features from the training data,
# and this is the input the model will use to learn patterns

X_test = test[features]
# Does the same for the test data, which we will later predict

# this way the model learns a function:
# revenue ≈ f(budget, popularity, runtime, genres, cast)

# set goal variable

# revenue values are extremely skewed because blockbuster movies can earn billions
# log transformation helps reduce the effect of huge outliers
y = np.log1p(data["revenue"])

# solve missing values

# replaces all missing values (NaN) with 0
# ml models cannot handle missing values directly
X = X.fillna(0)
X_test = X_test.fillna(0)

# create model

# Random Forest builds many decision trees and averages them
# tuned parameters can reduce overfitting and improve generalization
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# here we create the regression model,
# but at this point, the model exists but hasn’t learned anything yet

# train model
model.fit(X, y)

# fit here means:
# tell the model:"Learn patterns from X to predict y"

# Random Forest builds many decision trees and averages their predictions,
# and now the model has learned correlations like:
# higher budget -> often higher revenue

# cross validation

# cross_val_score splits the data into multiple folds (like 5 subsets)
# it trains on 4 folds and tests on 1 fold, then rotates
# gives a more reliable estimate of model performance

scores = cross_val_score(
    model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

# scoring="neg_root_mean_squared_error"
# measures the average prediction error

# negative sign is how sklearn handles scoring
# they want scores to be maximized internally,
# but RMSE should actually be minimized

print("RMSE:", abs(scores.mean()))

# here the result is negative

# calculates the average cross-validation score
# shows how well the model predicts unseen data

# lower RMSE = better predictions

# feature importance
# shows which features the model considered most useful

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})



# make predictions

# model.predict returns predictions in log-space
predictions = np.expm1(model.predict(X_test))

# np.expm1 reverses the log transformation
# converts predictions back into real revenue values

# create submission DataFrame

submission = pd.DataFrame({
    "id": test["id"],
    "revenue": predictions
})

# save to csv
submission.to_csv("submission.csv", index=False)

   id                              belongs_to_collection    budget  \
0   1  [{'id': 313576, 'name': 'Hot Tub Time Machine ...  14000000   
1   2  [{'id': 107674, 'name': 'The Princess Diaries ...  40000000   
2   3                                                NaN   3300000   
3   4                                                NaN   1200000   
4   5                                                NaN         0   

                                              genres  \
0                     [{'id': 35, 'name': 'Comedy'}]   
1  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   
2                      [{'id': 18, 'name': 'Drama'}]   
3  [{'id': 53, 'name': 'Thriller'}, {'id': 18, 'n...   
4  [{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...   

                            homepage    imdb_id original_language  \
0                                NaN  tt2637294                en   
1                                NaN  tt0368933                en   
2  http://sonyclassics.com/whiplash